# Prompt Evaluation Pipeline with Gemini

This notebook implements an automated evaluation pipeline using Google Gemini. It generates a dataset of tasks, executes them, and performs a dual evaluation (syntax + model review).

### 1. Setup and Initialization
We import necessary libraries, including `pydantic` for structured data validation and the Google GenAI SDK. We also load the API key from the environment.

In [ ]:
import os
import json
import re
import ast
import sys
from pathlib import Path
from statistics import mean
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pydantic import BaseModel

project_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "requirements.txt").exists()),
    Path.cwd(),
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.gemini_retry import generate_content_with_retry

load_dotenv()

client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
MODEL_ID = "gemini-2.5-flash"

### 2. Structured Output Schemas
Using Pydantic, we define the exact structure we expect from Gemini. This allows the SDK to automatically validate and parse the JSON response into Python objects, eliminating the need for manual string cleaning.

In [2]:
# Define Structured Output Schemas
class Task(BaseModel):
    task: str
    format: str # 'python', 'json', or 'regex'

class Dataset(BaseModel):
    tasks: list[Task]

class Evaluation(BaseModel):
    strengths: list[str]
    weaknesses: list[str]
    reasoning: str
    score: float

### 3. Generation and Execution Logic
- `generate_dataset`: Uses the `Dataset` schema to produce a list of tasks.
- `run_prompt`: Executes the specific task and requests raw code output without markdown blocks.

Gemini calls in this notebook now use the shared Tenacity-based retry helper for `429` and `RESOURCE_EXHAUSTED` responses.

In [ ]:
def generate_dataset(count=3):
    prompt = f"""
    Generate an evaluation dataset for AWS-related tasks.
    Focus on tasks that require writing a single Python function, a single JSON object, or a regex.
    Generate {count} unique tasks.
    """
    
    response = generate_content_with_retry(
        client=client,
        model=MODEL_ID,
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=Dataset
        ),
    )
    return response.parsed.tasks

def run_prompt(test_case):
    prompt = f"""
    Solve the following task:
    {test_case['task']}
    
    * Respond ONLY with the requested {test_case['format']} code.
    * Do not include markdown blocks or commentary.
    """
    response = generate_content_with_retry(
        client=client,
        model=MODEL_ID,
        contents=prompt,
    )
    return response.text.strip()

### 4. Dual Validation Logic
- `validate_syntax`: A programmatic check to see if the code actually compiles or parses as valid JSON/Regex.
- `grade_by_model`: A semantic review where Gemini acts as an expert code reviewer to assess quality and correctness.

In [ ]:
def validate_syntax(text, format):
    try:
        if format == "json":
            json.loads(text)
        elif format == "python":
            ast.parse(text)
        elif format == "regex":
            re.compile(text)
        return 10
    except:
        return 0

def grade_by_model(test_case, output):
    prompt = f"""
    Evaluate this AWS-related solution.
    Task: {test_case['task']}
    Solution: {output}
    """
    response = generate_content_with_retry(
        client=client,
        model=MODEL_ID,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction="You are an expert AWS code reviewer.",
            response_mime_type="application/json",
            response_schema=Evaluation
        ),
    )
    return response.parsed

### 5. Main Execution Pipeline
This orchestrator runs the entire workflow: generating the dataset, executing tasks, grading them, and calculating an average score.

In [5]:
def run_eval():
    print("Generating dataset...")
    dataset = generate_dataset()
    
    # Save dataset to root data/ folder
    os.makedirs("../../data", exist_ok=True)
    with open("../../data/dataset_fns.json", "w") as f:
        json.dump([d.model_dump() for d in dataset], f, indent=2)
    print("Dataset saved to ../../data/dataset_fns.json")
    
    results = []
    
    for case in dataset:
        case_dict = case.model_dump()
        print(f"Running test case: {case_dict['task'][:50]}...")
        
        output = run_prompt(case_dict)
        syntax_score = validate_syntax(output, case_dict['format'])
        model_eval = grade_by_model(case_dict, output)
        
        avg_score = (syntax_score + model_eval.score) / 2
        
        results.append({
            "task": case_dict['task'],
            "output": output,
            "score": avg_score,
            "reasoning": model_eval.reasoning
        })
    
    print(f"\nEvaluation Complete. Average Score: {mean([r['score'] for r in results]):.2f}")
    
    # Save results to root data/ folder
    with open("../../data/results_fns.json", "w") as f:
        json.dump(results, f, indent=2)
    print("Results saved to ../../data/results_fns.json")
    
    return results

results = run_eval()
import json
print(json.dumps(results, indent=2))

Generating dataset...
Dataset saved to ../../data/dataset_fns.json
Running test case: Write a Python function `get_s3_object_content(buc...
Running test case: Generate an IAM policy JSON object that grants rea...
Running test case: Provide a regular expression that validates an AWS...

Evaluation Complete. Average Score: 7.08
Results saved to ../../data/results_fns.json
[
  {
    "task": "Write a Python function `get_s3_object_content(bucket_name, key)` using `boto3` that retrieves and returns the content of an S3 object as a string.",
    "output": "import boto3\n\ndef get_s3_object_content(bucket_name, key):\n    s3 = boto3.client('s3')\n    response = s3.get_object(Bucket=bucket_name, Key=key)\n    content_body = response['Body']\n    content_string = content_body.read().decode('utf-8')\n    return content_string",
    "score": 6.75,
    "reasoning": "The solution effectively accomplishes the core task of retrieving S3 object content and decoding it into a string. The use of `boto3.